**Matrix mechanics: infinite and finite square wells.**
In standard quantum mechanics the time-independent Schrödinger equation (TISE)
is a continuous differential equation:

$$
\hat{H}\,\psi(x) = E\,\psi(x)
\qquad\text{where}\qquad
\hat{H} = -\frac{\hbar^2}{2m}\frac{d^2}{dx^2} + V(x)
$$

**Matrix mechanics** replaces this with a purely algebraic eigenvalue problem
by discretizing space into $N$ equally spaced grid points separated by $\Delta x$.
The continuous wavefunction $\psi(x)$ becomes a vector of $N$ values,
and the Hamiltonian operator $\hat{H}$ becomes an $N \times N$ matrix.
Solving the TISE reduces to diagonalizing that matrix.

This notebook solves two versions of the 1-D square well back to back
using identical grids so the results can be compared directly:

- **Infinite square well** ($V = \infty$ outside): wavefunctions are
  strictly zero at and beyond the walls. No tunnelling. Energies are
  exactly $E_n = n^2\pi^2\hbar^2/(2mL^2)$.
- **Finite square well** ($V = V_0$ outside): wavefunctions penetrate
  exponentially into the classically forbidden region. Energies are
  slightly lower than the infinite-well values.

In [ ]:
"""square_well.ipynb"""

# Cell 01 - Shared physical parameters and spatial grid
# These parameters are used by both the infinite and finite well sections.

import matplotlib.pyplot as plt
import numpy as np

# Physical well dimensions (natural units)
L = 100  # well width: always spans 0 to L regardless of dx
L_left = 15  # physical extent of classically forbidden region on each side
L_right = 15

# Grid resolution: reduce dx for finer sampling of the wavefunctions
dx = 0.25  # spatial step size

# Derive grid point counts from physical dimensions and step size
N_well = int(L / dx)
N_left = int(L_left / dx)
N_right = int(L_right / dx)

# Physical constants (natural units: hbar = m = 1)
hbar = 1.0
m = 1.0
V0 = 0.05  # finite well barrier height (used only in the finite well section)

# Build the spatial grid: x runs from -L_left to L + L_right in steps of dx
x = np.arange(-N_left, N_well + N_right + 1) * dx
left_wall = 0.0
right_wall = float(L)

# Finite difference coefficient for kinetic energy (same for both wells)
beta = hbar**2 / (2.0 * m * dx**2)

print(f"Well width L         : {L} (natural units)")
print(f"Step size dx         : {dx}")
print(f"Interior grid points : {N_well}")
print(f"Total grid points    : {len(x)}")
print(f"Barrier height V0    : {V0}  (finite well only)")
print(f"beta                 : {beta}")

**Discretizing the kinetic energy operator.**
The second derivative in the kinetic energy term is approximated by the
**centered finite difference**:

$$
\frac{d^2\psi}{dx^2}\bigg|_i
\approx
\frac{\psi_{i-1} - 2\psi_i + \psi_{i+1}}{(\Delta x)^2}
$$

Substituting into $-\dfrac{\hbar^2}{2m}\dfrac{d^2\psi}{dx^2}$ and defining
$\beta = \dfrac{\hbar^2}{2m(\Delta x)^2}$, row $i$ of the Hamiltonian
matrix contributes:

$$
H_{ii} = 2\beta + V_i
\qquad
H_{i,\,i\pm1} = -\beta
$$

The result is a **symmetric tridiagonal matrix** - diagonal entries hold
kinetic + potential energy at each grid point, and off-diagonal entries
couple each point to its two nearest neighbors.
For a five-point grid the structure looks like:

$$
H = \begin{bmatrix}
2\beta+V_1 & -\beta & 0 & 0 & 0 \\
-\beta & 2\beta+V_2 & -\beta & 0 & 0 \\
0 & -\beta & 2\beta+V_3 & -\beta & 0 \\
0 & 0 & -\beta & 2\beta+V_4 & -\beta \\
0 & 0 & 0 & -\beta & 2\beta+V_5
\end{bmatrix}
$$

**Dirichlet boundary conditions** ($\psi = 0$ at both endpoints) are
enforced by working only with the $N$ **interior** points and discarding
the endpoints - their zero values contribute nothing to the matrix.
For the infinite well $V_i = 0$ inside, so all diagonal entries equal $2\beta$.
For the finite well $V_i = V_0$ in the barrier regions, raising those
diagonal entries to $2\beta + V_0$.

---
**Part 1 - Infinite Square Well**

The infinite square well sets $V = \infty$ outside the well, which in the
matrix formulation means the wavefunction is forced to exactly zero at
the two boundary points.
This is already enforced by the Dirichlet boundary conditions - we simply
work only with the interior grid points and set $V = 0$ everywhere,
so the Hamiltonian contains only kinetic energy terms.

The analytic solution gives exact energy levels:

$$
E_n^\infty = \frac{n^2\pi^2\hbar^2}{2mL^2}, \quad n = 1, 2, 3, \ldots
$$

These serve as a verification of the matrix method before introducing
the finite well.

In [ ]:
# Cell 02 - Build and diagonalize the infinite well Hamiltonian
# V = 0 everywhere inside; boundary conditions enforce psi = 0 at the walls.

N_inf = N_well  # only interior well points (no barrier region needed)

diag_inf = 2.0 * beta * np.ones(N_inf)  # pure kinetic energy, V=0 inside
off_inf = -beta * np.ones(N_inf - 1)
H_inf = np.diag(diag_inf) + np.diag(off_inf, 1) + np.diag(off_inf, -1)

energies_inf, vecs_inf = np.linalg.eigh(H_inf)

print(f"Infinite well matrix shape : {H_inf.shape}")
print()
print("First five energy levels (numerical vs. analytic):")
for n in range(5):
    E_analytic = (n + 1) ** 2 * np.pi**2 * hbar**2 / (2 * m * L**2)
    print(
        f"  n={n + 1}: numerical={energies_inf[n]:.8f}  "
        f"analytic={E_analytic:.8f}  "
        f"diff={abs(energies_inf[n] - E_analytic):.2e}"
    )

**Infinite well wavefunctions.**
For the infinite well the grid only spans the interior of the well
(0 to $L$) - there are no exterior points because the wavefunction is
exactly zero everywhere outside by definition.
All eigenstates are valid; we plot the four lowest and the highest
of the states we solved.

In [ ]:
# Cell 03 - Plot infinite well probability densities

# Interior x grid for the infinite well (no exterior padding needed)
x_inf = np.arange(1, N_inf + 1) * dx  # positions of interior points

# Pad with zeros at the two boundary points to show psi=0 at walls
x_inf_full = np.concatenate([[0.0], x_inf, [L]])

plot_indices_inf = list(range(4)) + [N_inf - 1]  # first 4 + last

plt.figure("infinite_square_well", figsize=(10, 6))

for n in plot_indices_inf:
    psi_int = vecs_inf[:, n]
    psi_int = psi_int / np.sqrt(np.sum(np.abs(psi_int) ** 2) * dx)
    psi_full = np.concatenate([[0.0], psi_int, [0.0]])
    plt.plot(
        x_inf_full, np.abs(psi_full) ** 2, label=f"n={n + 1},  E={energies_inf[n]:.5f}"
    )

plt.axvspan(left_wall, right_wall, alpha=0.12, color="steelblue", label="well region")
plt.axhline(0, color="black", linewidth=0.8)
plt.xlabel("Position $x$ (natural units)")
plt.ylabel(r"Probability density $|\psi_n(x)|^2$")
plt.title(
    f"Infinite Square Well: 4 Lowest + Highest State\n"
    f"($L={L}$, $\\hbar=m=1$, $\\Delta x={dx}$, matrix size $N={N_inf}$)"
)
plt.legend(loc="upper right")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**Reading the infinite well plot.**
The infinite well has several features that distinguish it from the
finite well:

- **Hard walls:** probability density drops to exactly zero at $x=0$
  and $x=L$. There is zero probability of finding the electron outside
  the well - no tunnelling.
- **Exact analytic match:** the numerical energies agree with
  $E_n = n^2\pi^2\hbar^2/(2mL^2)$ to many decimal places, confirming
  the matrix method is working correctly.
- **Node structure:** state $n$ has $n-1$ nodes (zero crossings) inside
  the well and exactly $n$ peaks.

---
**Part 2 - Finite Square Well**

The finite square well sets $V = V_0$ outside the well instead of
$V = \infty$.
The grid now extends into the classically forbidden regions on both sides
so the wavefunction's exponential decay into the barrier is visible.
Only states with $E_n < V_0$ are **bound states**; higher eigenstates are
scattering states that are an artifact of the finite grid.

The potential is:

$$
V(x) = \begin{cases} 0 & 0 \leq x \leq L \\ V_0 & \text{otherwise} \end{cases}
$$

In [ ]:
# Cell 04 - Build and diagonalize the finite well Hamiltonian

# Potential: 0 inside the well, V0 in the classically forbidden regions
V_fin = np.where((x >= left_wall) & (x <= right_wall), 0.0, V0)

# Interior points only (Dirichlet BCs at the grid endpoints)
V_int = V_fin[1:-1]
N_fin = len(V_int)

diag_fin = 2.0 * beta + V_int
off_fin = -beta * np.ones(N_fin - 1)
H_fin = np.diag(diag_fin) + np.diag(off_fin, 1) + np.diag(off_fin, -1)

energies_fin, vecs_fin = np.linalg.eigh(H_fin)

bound_mask = energies_fin < V0
n_bound = np.sum(bound_mask)

print(f"Finite well matrix shape    : {H_fin.shape}")
print(f"Bound states (E < V0={V0}) : {n_bound}")
print()
print("First five bound-state energies (finite vs. infinite well):")
for n in range(5):
    E_inf = (n + 1) ** 2 * np.pi**2 * hbar**2 / (2 * m * L**2)
    print(
        f"  n={n + 1}: finite={energies_fin[n]:.8f}  "
        f"infinite={E_inf:.8f}  "
        f"diff={E_inf - energies_fin[n]:.2e}"
    )

**Finite well wavefunctions.**
The grid now includes the classically forbidden regions on both sides,
so the full wavefunction - including its exponential tail into the
barrier - is visible.
We select the four lowest bound states and the highest bound state.

In [ ]:
# Cell 05 - Plot finite well probability densities

bound_indices = np.where(energies_fin < V0)[0]
plot_indices_fin = list(bound_indices[:4]) + [bound_indices[-1]]

plt.figure("finite_square_well", figsize=(10, 6))

for n in plot_indices_fin:
    psi_int = vecs_fin[:, n]
    psi_int = psi_int / np.sqrt(np.sum(np.abs(psi_int) ** 2) * dx)
    psi = np.zeros_like(x, dtype=float)
    psi[1:-1] = psi_int
    plt.plot(x, np.abs(psi) ** 2, label=f"n={n + 1},  E={energies_fin[n]:.5f}")

plt.axvspan(
    left_wall, right_wall, alpha=0.12, color="steelblue", label="well region (V=0)"
)
plt.axhline(0, color="black", linewidth=0.8)
plt.xlabel("Position $x$ (natural units)")
plt.ylabel(r"Probability density $|\psi_n(x)|^2$")
plt.title(
    f"Finite Square Well: 4 Lowest + Highest Bound State\n"
    f"($L={L}$, $V_0={V0}$, $\\hbar=m=1$, $\\Delta x={dx}$, matrix size $N={N_fin}$)"
)
plt.legend(loc="upper right")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**Reading the finite well plot.**
Comparing this plot to the infinite well reveals the physical
consequences of a finite barrier:

- **Barrier penetration:** the wavefunctions do not drop to zero at the
  well walls - they decay exponentially into the shaded region,
  giving a non-zero probability of finding the electron in the
  classically forbidden zone. This is quantum tunnelling.
- **Lower energies:** finite well energies are slightly lower than the
  infinite-well values because the electron can spread into the barrier,
  effectively experiencing a larger box.
- **Finite number of bound states:** unlike the infinite well which has
  infinitely many bound states, the finite well supports only $n_{\text{bound}}$
  states with $E_n < V_0$. Higher eigenstates are scattering states.
- **Node structure:** state $n$ still has $n-1$ nodes inside the well,
  but the wavefunction no longer vanishes at the walls.

**What the matrix did:** a single call to `np.linalg.eigh` on the
tridiagonal Hamiltonian solved all bound states simultaneously,
with no need for the transcendental equations that the analytic
finite-well solution requires.

---
**Part 3 - Higher-Order Accuracy: 4th-Order Finite Differences**

The standard centered difference used in Parts 1 and 2 is **2nd-order accurate**
- its error in the second derivative scales as $(\Delta x)^2$.
A higher-order 5-point stencil for the second derivative is **4th-order accurate**,
with error scaling as $(\Delta x)^4$:

$$
\frac{d^2\psi}{dx^2}\bigg|_i
\approx
\frac{-\psi_{i-2} + 16\psi_{i-1} - 30\psi_i + 16\psi_{i+1} - \psi_{i+2}}
{12\,(\Delta x)^2}
$$

Substituting into $T = -\dfrac{\hbar^2}{2m}\dfrac{d^2\psi}{dx^2}$ and using
$\beta = \dfrac{\hbar^2}{2m(\Delta x)^2}$ gives three distinct matrix entry types:

$$
H_{ii} = \tfrac{5}{2}\beta + V_i
\qquad
H_{i,\,i\pm1} = -\tfrac{4}{3}\beta
\qquad
H_{i,\,i\pm2} = \tfrac{1}{12}\beta
$$

This produces a **symmetric pentadiagonal matrix** - each row couples
to its four nearest neighbors instead of two.
For a six-point grid the structure looks like:

$$
H = \begin{bmatrix}
\tfrac{5}{2}\beta+V_1 & -\tfrac{4}{3}\beta & \tfrac{1}{12}\beta & 0 & 0 & 0 \\
-\tfrac{4}{3}\beta & \tfrac{5}{2}\beta+V_2 & -\tfrac{4}{3}\beta & \tfrac{1}{12}\beta & 0 & 0 \\
\tfrac{1}{12}\beta & -\tfrac{4}{3}\beta & \tfrac{5}{2}\beta+V_3 & -\tfrac{4}{3}\beta & \tfrac{1}{12}\beta & 0 \\
0 & \tfrac{1}{12}\beta & -\tfrac{4}{3}\beta & \tfrac{5}{2}\beta+V_4 & -\tfrac{4}{3}\beta & \tfrac{1}{12}\beta \\
0 & 0 & \tfrac{1}{12}\beta & -\tfrac{4}{3}\beta & \tfrac{5}{2}\beta+V_5 & -\tfrac{4}{3}\beta \\
0 & 0 & 0 & \tfrac{1}{12}\beta & -\tfrac{4}{3}\beta & \tfrac{5}{2}\beta+V_6
\end{bmatrix}
$$

`np.linalg.eigh` handles this without any change since it works on any
real symmetric matrix.
The analytic energies of the infinite square well
$E_n = n^2\pi^2\hbar^2/(2mL^2)$ provide the ground truth to measure
relative error for both methods across all energy levels.

In [ ]:
# Cell 06 - Build the 4th-order pentadiagonal Hamiltonian and compare accuracy
# Uses the infinite well (V=0 everywhere inside) so analytic energies are exact.

# 4th-order stencil coefficients
alpha_0 = (5 / 2) * beta  # main diagonal kinetic coefficient
alpha_1 = (-4 / 3) * beta  # nearest-neighbor coupling
alpha_2 = (1 / 12) * beta  # next-nearest-neighbor coupling

print(f"2nd-order diagonal coeff      : {2 * beta:.6f}")
print(f"4th-order diagonal coeff      : {alpha_0:.6f}")
print(f"4th-order off-diag +/-1 coeff : {alpha_1:.6f}")
print(f"4th-order off-diag +/-2 coeff : {alpha_2:.6f}")

# Build the pentadiagonal Hamiltonian (infinite well: V=0 inside)
N4 = N_inf  # same number of interior points as the infinite well
H4 = (
    np.diag(alpha_0 * np.ones(N4))
    + np.diag(alpha_1 * np.ones(N4 - 1), 1)
    + np.diag(alpha_1 * np.ones(N4 - 1), -1)
    + np.diag(alpha_2 * np.ones(N4 - 2), 2)
    + np.diag(alpha_2 * np.ones(N4 - 2), -2)
)

energies_4th = np.linalg.eigh(H4)[0]

print(f"\nMatrix shape (pentadiagonal)   : {H4.shape}")

**Comparing relative errors across energy levels.**
The analytic infinite-well energies $E_n = n^2\pi^2\hbar^2/(2mL^2)$
are used as the reference.
The relative error for each level is:

$$
\epsilon_n = \frac{|E_n^{\text{numerical}} - E_n^{\text{analytic}}|}{E_n^{\text{analytic}}} \times 100\%
$$

Both methods use identical grids and the same `eigh` solver - the only
difference is the stencil width.
The 4th-order curve should sit consistently below the 2nd-order curve,
and both should increase with $n$ because higher states have shorter
wavelengths that are harder to resolve on a fixed grid.

In [ ]:
# Cell 07 - Plot relative error vs energy level for both methods

n_compare = 20  # number of energy levels to compare
levels = np.arange(1, n_compare + 1)

E_analytic = np.array([(n) ** 2 * np.pi**2 * hbar**2 / (2 * m * L**2) for n in levels])

err_2nd = np.abs(energies_inf[:n_compare] - E_analytic) / E_analytic * 100
err_4th = np.abs(energies_4th[:n_compare] - E_analytic) / E_analytic * 100

plt.figure("accuracy_comparison", figsize=(10, 5))
plt.plot(levels, err_2nd, marker="o", label="2nd-order (tridiagonal)")
plt.plot(levels, err_4th, marker="s", label="4th-order (pentadiagonal)")
plt.xlabel("Energy level $n$")
plt.ylabel("Relative error (%)")
plt.title(
    f"Eigenvalue Accuracy: 2nd-Order vs 4th-Order Finite Differences\n"
    f"(Infinite square well, $L={L}$, $\\Delta x={dx}$, $N={N_inf}$ interior points)"
)
plt.xticks(levels)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(
    f"{'n':>3}  {'E_analytic':>14}  {'err_2nd %':>12}  {'err_4th %':>12}  {'improvement':>12}"
)
print("-" * 60)
for i, n in enumerate(levels):
    print(
        f"{n:3d}  {E_analytic[i]:14.8f}  {err_2nd[i]:12.8f}  "
        f"{err_4th[i]:12.8f}  {err_2nd[i] / err_4th[i]:10.2f}x"
    )